# 🛡️ DeepShield Audio — Notebook 02: Preprocessing Walkthrough

This notebook demonstrates the preprocessing pipeline:
- Loading audio and checking sample rates
- Padding and trimming to the target max length (4 seconds)
- Extracting Log-Mel Spectrograms
- Normalisation (Standardisation)
- Visualising the transformations

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import librosa
from pathlib import Path

from src.config import SAMPLE_RATE, MAX_AUDIO_SECONDS, MAX_AUDIO_SAMPLES, N_MELS, T_FRAMES
from src.preprocessor import load_audio, pad_or_trim, extract_log_mel, normalise_spectrogram, preprocess_file
from src.utils import plot_waveform, plot_spectrogram
from src.data_parser import load_split

plt.rcParams['figure.facecolor'] = '#1E1E2E'
plt.rcParams['axes.facecolor'] = '#1E1E2E'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

print('✅ Imports successful')

## 1. Load a Sample Audio File
Let's select a bonafide sample from the training split (if available) or generate a synthetic one for demonstration.

In [ ]:
import os
from src.data_parser import make_synthetic_dataframe

# Try to load real data, fallback to synthetic for demo
try:
    train_df = load_split('train')
    real_audio = train_df[train_df['label_str'] == 'bonafide']['file_path'].values[0]
except (FileNotFoundError, IndexError):
    print('Real dataset not found, generating a synthetic sine wave...')
    # Generate synthetic sine wave audio file
    import soundfile as sf
    real_audio = '/tmp/demo_audio.wav'
    t = np.linspace(0, 3.5, int(3.5 * SAMPLE_RATE))
    wav = (np.sin(2 * np.pi * 440 * t) + 0.5 * np.sin(2 * np.pi * 880 * t)).astype(np.float32)
    sf.write(real_audio, wav, SAMPLE_RATE)

print(f'Using audio file: {real_audio}')

# Step 1: Load audio
waveform = load_audio(real_audio)
print(f'Original Waveform Shape: {waveform.shape}')
print(f'Duration: {len(waveform)/SAMPLE_RATE:.2f} seconds')

fig, ax = plt.subplots(figsize=(10, 2))
plot_waveform(waveform, ax=ax, title='Original Waveform')
plt.show()

## 2. Pad or Trim to Fixed Length
Neural networks require fixed-size inputs (unless using fully convolutional/recurrent approaches with dynamic batching). We pad short files with zeros or trim long files.

In [ ]:
# Step 2: Pad/Trim
padded_waveform = pad_or_trim(waveform, MAX_AUDIO_SAMPLES)
print(f'Target Max Samples: {MAX_AUDIO_SAMPLES}')
print(f'Padded Waveform Shape: {padded_waveform.shape}')

fig, ax = plt.subplots(figsize=(10, 2))
plot_waveform(padded_waveform, ax=ax, title='Padded / Trimmed Waveform')
plt.show()

## 3. Extract Log-Mel Spectrogram
Convert the 1D waveform into a 2D time-frequency representation. This highlights perceptually relevant acoustic features.

In [ ]:
# Step 3: Extract Log-Mel
spec = extract_log_mel(padded_waveform)
print(f'Spectrogram Shape: {spec.shape} (N_MELS x T_FRAMES)')
print(f'Min dB: {spec.min():.2f}, Max dB: {spec.max():.2f}')

fig, ax = plt.subplots(figsize=(10, 4))
plot_spectrogram(spec, title='Log-Mel Spectrogram (Unnormalised)', ax=ax)
plt.show()

## 4. Normalisation
Deep learning models train better with zero-mean, unit-variance inputs. We apply global normalisation based on training set statistics.

In [ ]:
# Step 4: Normalise
norm_spec = normalise_spectrogram(spec, mean=-35.0, std=15.0)  # Dummy stats for demo
print(f'Normalised Spectrogram Min: {norm_spec.min():.2f}, Max: {norm_spec.max():.2f}')

fig, ax = plt.subplots(figsize=(10, 4))
# Note: colormap limits are different after normalisation
img = librosa.display.specshow(norm_spec, sr=SAMPLE_RATE, hop_length=256, x_axis='time', y_axis='mel', ax=ax, cmap='magma')
fig.colorbar(img, ax=ax, format='%+2.1f')
ax.set_title('Normalised Spectrogram', color='white')
plt.show()

## 5. Full Pipeline Output (`preprocess_file`)
The full pipeline encapsulates all these steps and adds the channel dimension `(128, 251, 1)` required by TensorFlow's Conv2D layers.

In [ ]:
final_input = preprocess_file(real_audio, add_channel_dim=True)
print(f'Final Model Input Shape: {final_input.shape}')
print(f'Data Type: {final_input.dtype}')
print('Ready for model training / inference!')